# Motor Overheating Prediction — v4: Best-of-Both Pipeline

**Combina lo mejor de v2 y v3:**
- **Pipeline de datos y features de v3**: trust-filter, baselines robustos (cummin/quantile), matched filter shape-aware, features cross-motor, inyeccion calibrada por motor, GroupKFold, rate matching anclado a prevalencia real, Viterbi.
- **Seleccion de algoritmo de v2**: para cada motor se comparan **5 algoritmos** (LR, XGBoost, RF, HGB, IsolationForest) y gana el de mayor F1 OOF con rate matching.

| Seccion | Contenido |
|---------|----------|
| § 1 | Entorno |
| § 2 | Imports |
| § 3 | Preprocessing minimo + trust-filter de additional_data |
| § 4 | Feature engineering (shape, baselines robustos, cross-motor) |
| § 5 | Inyeccion calibrada por motor (motores 3 y 5: sutil 1-4C) |
| § 6 | Config: factoria de modelos, rate matching, Viterbi |
| § 7a-e | Carga, filtrado, features, inyeccion |
| § 8 | train_motor() — compara LR/XGB/RF/HGB/IsoForest por motor |
| § 9-14 | Entrenamiento motor a motor (celdas independientes) |
| § 15 | Resumen: ganador y OOF F1 por motor |
| § 16 | Submission con rate matching y Viterbi |

## § 1 · Environment verification

Checks the Python kernel and that all required data paths exist before proceeding.

In [1]:
import sys, os

print(f'Python : {sys.version}')
print(f'Kernel : {sys.executable}')
print()

NOTEBOOK_ROOT   = os.getcwd()
DATA_ROOT       = os.path.join(NOTEBOOK_ROOT, 'kaggle_data_challenge', 'kaggle_data_challenge')
TRAIN_PATH      = os.path.join(DATA_ROOT, 'training_data') + os.sep
TEST_PATH       = os.path.join(DATA_ROOT, 'testing_data')  + os.sep
SAMPLE_SUB_PATH = os.path.join(DATA_ROOT, 'sample_submission.csv')
ADDITIONAL_BASE = os.path.join(NOTEBOOK_ROOT, 'additional_data')
UTILITY_PATH    = os.path.join(NOTEBOOK_ROOT, 'kaggle_data_challenge')

checks = {
    'utility.py'                  : os.path.join(UTILITY_PATH, 'utility.py'),
    'training_data/'              : TRAIN_PATH,
    'testing_data/'               : TEST_PATH,
    'sample_submission.csv'       : SAMPLE_SUB_PATH,
    'additional_data/ (opcional)' : ADDITIONAL_BASE,
}
all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    mark   = 'OK   ' if exists else 'FALTA'
    print(f'  {mark}  {label:38s}  {path}')
    if not exists and 'opcional' not in label:
        all_ok = False

if os.path.exists(ADDITIONAL_BASE):
    grupos = [d for d in os.listdir(ADDITIONAL_BASE)
              if os.path.isdir(os.path.join(ADDITIONAL_BASE, d))
              and not d.startswith('_tmp')]
    print(f'\n  Grupos en additional_data/ ({len(grupos)} encontrados):')
    for g in grupos:
        gp      = os.path.join(ADDITIONAL_BASE, g)
        subdirs = [d for d in os.listdir(gp) if os.path.isdir(os.path.join(gp, d))]
        has_xl  = os.path.exists(os.path.join(gp, 'Test conditions.xlsx'))
        has_cp  = os.path.exists(os.path.join(gp, 'Test conditions copy.xlsx'))
        xl_mark = 'xlsx OK' if has_xl else ('xlsx (copia)' if has_cp else 'SIN xlsx')
        print(f'    {g}  —  {len(subdirs)} secuencias  —  {xl_mark}')

print()
print('Entorno OK' if all_ok else 'ATENCION: faltan rutas criticas')

Python : 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
Kernel : c:\Users\oscar\Desktop\TDS INDUSTRY\.venv\Scripts\python.exe

  OK     utility.py                              c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\utility.py
  OK     training_data/                          c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\kaggle_data_challenge\training_data\
  OK     testing_data/                           c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\kaggle_data_challenge\testing_data\
  OK     sample_submission.csv                   c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\kaggle_data_challenge\sample_submission.csv
  OK     additional_data/ (opcional)             c:\Users\oscar\Desktop\TDS INDUSTRY\additional_data

  Grupos en additional_data/ (3 encontrados):
    additional_data_20240524_group_6  —  14 secuencias  —  xlsx OK
    additional_training_data_group_1  —  13 secuencias  —  xlsx OK
 

## § 2 · Imports

All imports for the pipeline, including RandomForestClassifier, HistGradientBoostingClassifier,
GroupKFold and numpy stride tricks for the matched filter.

In [2]:
import warnings, shutil
import numpy  as np
import pandas as pd

from numpy.lib.stride_tricks import sliding_window_view

import xgboost as xgb
from sklearn.ensemble        import (RandomForestClassifier,
                                     HistGradientBoostingClassifier,
                                     IsolationForest)
from sklearn.linear_model    import LogisticRegression
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics         import f1_score

warnings.filterwarnings('ignore')

import sys, os
sys.path.insert(1, UTILITY_PATH)
from utility import read_all_test_data_from_path

print('All imports OK')
print(f'XGBoost {xgb.__version__}')

All imports OK
XGBoost 3.2.0


## § 3 · Helper functions

### `pre_processing_minimal`
Only physical clip + forward-fill. NO centering, NO derived features.
Robust baselines handle centering better.

### `trust_filter_additional`
For each (test_condition, motor) block that is labelled as faulty:
- Compute baseline = min temperature during normal periods in that sequence.
- If max fault temperature does NOT exceed baseline + margin → void the labels (set to 0).
- Returns (filtered_df, audit_df) showing kept / voided counts per block.

In [3]:
# ─────────────────────────────────────────────────────────────────
#  Minimal preprocessing callback (used by read_all_test_data_from_path)
# ─────────────────────────────────────────────────────────────────
def pre_processing_minimal(df: pd.DataFrame):
    """Physical clip + ffill only. No centering, no feature derivation."""
    if len(df) == 0:
        return
    df['temperature'] = (
        df['temperature']
        .where(df['temperature'].between(0, 100), np.nan)
        .ffill()
    )
    df['voltage'] = (
        df['voltage']
        .where(df['voltage'].between(6000, 9000), np.nan)
        .ffill()
    )
    df['position'] = (
        df['position']
        .where(df['position'].between(0, 1000), np.nan)
        .ffill()
    )


# ─────────────────────────────────────────────────────────────────
#  Trust-filter for additional_data
# ─────────────────────────────────────────────────────────────────
def trust_filter_additional(df: pd.DataFrame, temp_rise_margin: float = 1.0):
    """
    For each (test_condition, motor) block labelled as faulty:
      - baseline = min temperature during *normal* (label=0) periods in the same sequence.
      - If max fault temperature <= baseline + temp_rise_margin  →  void labels (set 0).

    Returns
    -------
    filtered_df : DataFrame with voided labels set to 0
    audit_df    : DataFrame with columns [test_condition, motor_id, kept, voided]
    """
    df = df.copy()
    audit_rows = []

    for seq in df['test_condition'].unique():
        seq_mask = df['test_condition'] == seq
        seq_df   = df[seq_mask]

        for mid in range(1, 7):
            label_col = f'data_motor_{mid}_label'
            temp_col  = f'data_motor_{mid}_temperature'

            if label_col not in seq_df.columns or temp_col not in seq_df.columns:
                continue

            labels = seq_df[label_col]
            temps  = seq_df[temp_col]

            # Only process if there are labelled faults
            if (labels == 1).sum() == 0:
                continue

            normal_temps = temps[labels == 0]
            if len(normal_temps) == 0:
                continue
            baseline     = float(normal_temps.min())
            fault_temps  = temps[labels == 1]
            max_fault_t  = float(fault_temps.max())

            if max_fault_t <= baseline + temp_rise_margin:
                # Void the labels — mislabelled sequence
                df.loc[seq_mask & (df[label_col] == 1), label_col] = 0
                audit_rows.append({
                    'test_condition': seq, 'motor_id': mid,
                    'kept': 0, 'voided': int((labels == 1).sum()),
                    'baseline': round(baseline, 3),
                    'max_fault_temp': round(max_fault_t, 3),
                })
            else:
                audit_rows.append({
                    'test_condition': seq, 'motor_id': mid,
                    'kept': int((labels == 1).sum()), 'voided': 0,
                    'baseline': round(baseline, 3),
                    'max_fault_temp': round(max_fault_t, 3),
                })

    audit_df = pd.DataFrame(audit_rows)
    return df, audit_df


# ─────────────────────────────────────────────────────────────────
#  Group loader (same logic as v2 but uses pre_processing_minimal)
# ─────────────────────────────────────────────────────────────────
def _validate_sequence(seq_path):
    csvs = [f for f in os.listdir(seq_path) if f.endswith('.csv')]
    if not csvs:
        return False
    for csv_file in csvs:
        try:
            lines = [l for l in open(os.path.join(seq_path, csv_file)).readlines() if l.strip()]
            if len(lines) <= 1:
                return False
        except Exception:
            return False
    return True


def _load_group_raw(group_path, group_name):
    """Load one additional_data group with minimal preprocessing, return raw DataFrame or None."""
    xlsx      = os.path.join(group_path, 'Test conditions.xlsx')
    xlsx_copy = os.path.join(group_path, 'Test conditions copy.xlsx')
    if not os.path.exists(xlsx) and os.path.exists(xlsx_copy):
        try:
            shutil.copy(xlsx_copy, xlsx)
        except Exception:
            pass
    if not os.path.exists(xlsx):
        return None

    tmp = os.path.join(ADDITIONAL_BASE, f'_tmp_{group_name}')
    try:
        if os.path.exists(tmp):
            shutil.rmtree(tmp)
    except Exception:
        pass
    os.makedirs(tmp, exist_ok=True)
    shutil.copy(xlsx, os.path.join(tmp, 'Test conditions.xlsx'))

    subdirs = [d for d in os.listdir(group_path) if os.path.isdir(os.path.join(group_path, d))]
    n_ok = 0
    for sd in subdirs:
        sp  = os.path.join(group_path, sd)
        dst = os.path.join(tmp, sd)
        if _validate_sequence(sp):
            shutil.copytree(sp, dst, dirs_exist_ok=True)
            n_ok += 1

    result = None
    if n_ok > 0:
        try:
            result = read_all_test_data_from_path(tmp + os.sep, pre_processing_minimal, is_plot=False)
        except Exception as e:
            print(f'  [error loading {group_name}]: {e}')

    try:
        shutil.rmtree(tmp)
    except Exception:
        pass
    return result


print('Helper functions defined')

Helper functions defined


## § 4 · Feature engineering

### `_matched_filter(elev_arr, scales)`
Multi-scale matched filter using `numpy.lib.stride_tricks.sliding_window_view`.
For each scale L: builds a rise-decay template (rise L//3, fall 2L//3), zero-mean unit-norm.
Returns (best_corr, best_scale) per timestep.

### `_motor_features(temp, pos, volt, include_shape)`
All per-motor time-series features: deltas, rates, rolling stats, drift, CUSUM, run-length,
slope asymmetry, and (optionally) matched-filter outputs.

### `build_features_for_df(df)`
Operates on the wide DataFrame (all motors).
Groups by test_condition, calls `_motor_features` per sequence per motor,
then adds cross-motor features (voltage deviation from peers, sum-of-others velocity, etc.).

In [4]:
# ─────────────────────────────────────────────────────────────────
#  Multi-scale matched filter
# ─────────────────────────────────────────────────────────────────
def _matched_filter(elev_arr: np.ndarray, scales=(15, 30, 60, 120)):
    """
    Multi-scale matched filter on elevation signal.
    For each scale L:
      - Build rise-decay template: rise over L//3, fall over 2L//3
      - Zero-mean, unit-norm normalise template
      - Slide window of size L over elev_arr (padded with L-1 zeros)
      - Normalise each window (mean/std), compute dot product → clip at 0
    Returns (best_corr, best_scale) arrays — max across scales per timestep.
    """
    n = len(elev_arr)
    best_corr  = np.zeros(n, dtype=np.float32)
    best_scale = np.zeros(n, dtype=np.float32)

    for L in scales:
        n_rise = L // 3
        n_fall = L - n_rise
        tmpl   = np.concatenate([np.linspace(0, 1, n_rise), np.linspace(1, 0, n_fall)])
        tmpl   = tmpl - tmpl.mean()
        norm_t = np.linalg.norm(tmpl)
        if norm_t < 1e-8:
            continue
        tmpl = tmpl / norm_t

        # Pad signal so output length == n
        padded = np.concatenate([np.zeros(L - 1, dtype=np.float32), elev_arr.astype(np.float32)])
        # shape: (n, L)
        windows = sliding_window_view(padded, window_shape=L)

        w_mean = windows.mean(axis=1, keepdims=True)
        w_std  = windows.std(axis=1, keepdims=True) + 1e-8
        w_norm = (windows - w_mean) / w_std

        corr = np.clip(w_norm @ tmpl, 0, None)  # shape (n,)

        mask = corr > best_corr
        best_corr[mask]  = corr[mask]
        best_scale[mask] = float(L)

    return best_corr, best_scale


# ─────────────────────────────────────────────────────────────────
#  Per-motor feature dict
# ─────────────────────────────────────────────────────────────────
def _motor_features(temp: pd.Series, pos: pd.Series, volt: pd.Series,
                    include_shape: bool = True) -> dict:
    """
    Compute per-motor features.  All outputs are numpy arrays of len(temp).

    Parameters
    ----------
    temp, pos, volt   : per-motor Series (already clipped / ffilled)
    include_shape     : whether to run the matched filter

    Returns
    -------
    dict of {feature_name: np.ndarray}
    """
    n = len(temp)
    out = {}

    # Basic deltas / rates
    out['temp_delta']  = (temp - temp.iloc[0]).values.astype(np.float32)
    velocity           = pos.diff(1).fillna(0)
    acceleration       = velocity.diff(1).fillna(0)
    temp_rate          = temp.diff(1).fillna(0)
    temp_accel         = temp_rate.diff(1).fillna(0)
    volt_rate          = volt.diff(1).fillna(0)

    out['velocity']    = velocity.values.astype(np.float32)
    out['acceleration']= acceleration.values.astype(np.float32)
    out['temp_rate']   = temp_rate.values.astype(np.float32)
    out['temp_accel']  = temp_accel.values.astype(np.float32)
    out['volt_rate']   = volt_rate.values.astype(np.float32)

    # Rolling std features
    out['roll_std_pos_50']  = pos.rolling(50,  min_periods=1).std().fillna(0).values.astype(np.float32)
    out['roll_std_temp_50'] = temp.rolling(50, min_periods=1).std().fillna(0).values.astype(np.float32)
    out['roll_std_volt_50'] = volt.rolling(50, min_periods=1).std().fillna(0).values.astype(np.float32)

    # Temperature drift vs rolling mean
    out['temp_drift200'] = (
        temp - temp.rolling(200, min_periods=1).mean()
    ).values.astype(np.float32)

    # Is-moving (smoothed)
    out['is_moving'] = (
        (velocity.abs() > 0.5).astype(float)
        .rolling(20, min_periods=1).mean()
    ).values.astype(np.float32)

    # Temperature above rolling min-200
    out['temp_above_min200'] = (
        temp - temp.rolling(200, min_periods=1).min()
    ).values.astype(np.float32)

    # Temperature range over rolling 200
    roll200_max = temp.rolling(200, min_periods=1).max()
    roll200_min = temp.rolling(200, min_periods=1).min()
    out['temp_range200'] = (roll200_max - roll200_min).values.astype(np.float32)

    # Temperature above rolling 10th-percentile over 600 samples
    roll600_q10 = temp.rolling(600, min_periods=10).quantile(0.10)
    # fill early NaNs with expanding quantile
    exp_q10     = temp.expanding(min_periods=5).quantile(0.10)
    qbase       = roll600_q10.fillna(exp_q10)
    out['temp_above_qbase'] = (temp - qbase).fillna(0).values.astype(np.float32)

    # Temperature above expanding 10th-percentile
    out['temp_above_expq'] = (
        temp - temp.expanding(min_periods=5).quantile(0.10)
    ).fillna(0).values.astype(np.float32)

    # Temperature above cumulative minimum  (elevation)
    elev_cummin = (temp - temp.expanding().min()).fillna(0)
    out['temp_above_cummin'] = elev_cummin.values.astype(np.float32)

    # Normalised elevation
    first50_std = float(temp.iloc[:50].std()) if n >= 50 else float(temp.std())
    if np.isnan(first50_std) or first50_std < 1e-6:
        first50_std = 1e-6
    elev_norm = elev_cummin / (first50_std + 1e-6)
    out['temp_elev_norm'] = elev_norm.values.astype(np.float32)

    # One-sided CUSUM on elev_cummin
    slack = 0.5
    elev_arr = elev_cummin.values.astype(np.float64)
    cusum = np.zeros(n, dtype=np.float32)
    s = 0.0
    for i in range(n):
        if elev_arr[i] < slack:
            s = 0.0
        else:
            s = s + elev_arr[i] - slack
        cusum[i] = s
    out['cusum_up'] = cusum

    # Consecutive run-length where elev_cummin >= 1.0
    elev_run = np.zeros(n, dtype=np.float32)
    run = 0
    for i in range(n):
        if elev_arr[i] >= 1.0:
            run += 1
        else:
            run = 0
        elev_run[i] = run
    out['elev_runlen'] = elev_run

    # Slope asymmetry (lag=15)
    lag = 15
    elev_s = elev_cummin.values.astype(np.float32)
    slope_asym = np.zeros(n, dtype=np.float32)
    for i in range(2 * lag, n):
        left  = max(0.0, elev_s[i - lag]   - elev_s[i - 2 * lag])
        right = max(0.0, elev_s[i - lag]   - elev_s[i])
        slope_asym[i] = left * right
    out['slope_asym'] = slope_asym

    # Multi-scale matched filter (optional)
    if include_shape:
        shape_match, shape_scale = _matched_filter(elev_arr, scales=(15, 30, 60, 120))
        out['shape_match'] = shape_match.astype(np.float32)
        out['shape_scale'] = shape_scale.astype(np.float32)

    return out


# ─────────────────────────────────────────────────────────────────
#  Main feature-build function
# ─────────────────────────────────────────────────────────────────
SHAPE_MOTORS = {1, 2, 3, 5}   # include_shape=True for these motors

def build_features_for_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build rich per-motor + cross-motor features on a WIDE DataFrame.
    Groups by test_condition, processes each sequence independently.
    Writes new columns in-place and returns the enriched DataFrame.
    """
    df = df.copy()
    sequences = df['test_condition'].unique()
    n_seq = len(sequences)
    print(f'  Building features for {n_seq} sequences ...')

    for seq_idx, seq in enumerate(sequences):
        if (seq_idx + 1) % 10 == 0 or (seq_idx + 1) == n_seq:
            print(f'  ... {seq_idx + 1}/{n_seq}')
        mask = df['test_condition'] == seq
        idx  = df.index[mask]

        per_motor_feats = {}  # mid -> {feat_name -> arr}

        # ── Per-motor pass ───────────────────────────────────────
        for mid in range(1, 7):
            tc  = f'data_motor_{mid}_temperature'
            pc  = f'data_motor_{mid}_position'
            vc  = f'data_motor_{mid}_voltage'

            if tc not in df.columns:
                continue

            temp = df.loc[mask, tc]
            pos  = df.loc[mask, pc] if pc in df.columns else pd.Series(np.zeros(mask.sum()), index=idx)
            volt = df.loc[mask, vc] if vc in df.columns else pd.Series(np.zeros(mask.sum()), index=idx)

            include_shape = (mid in SHAPE_MOTORS)
            feats = _motor_features(temp, pos, volt, include_shape=include_shape)
            per_motor_feats[mid] = feats

            for feat_name, arr in feats.items():
                col = f'data_motor_{mid}_{feat_name}'
                df.loc[mask, col] = arr

        # ── Cross-motor features ─────────────────────────────────
        # mean voltage across all 6 motors
        volt_cols = [f'data_motor_{mid}_voltage' for mid in range(1, 7)
                     if f'data_motor_{mid}_voltage' in df.columns]
        if volt_cols:
            mean_voltage = df.loc[mask, volt_cols].mean(axis=1)
            df.loc[mask, 'mean_voltage'] = mean_voltage.values

            for mid in range(1, 7):
                vc = f'data_motor_{mid}_voltage'
                if vc in df.columns:
                    df.loc[mask, f'data_motor_{mid}_volt_dev_peers'] = (
                        df.loc[mask, vc].values - mean_voltage.values
                    )

        # sum of other 5 motors' velocity
        for mid in range(1, 7):
            other_vel_cols = [
                f'data_motor_{m}_velocity' for m in range(1, 7)
                if m != mid and f'data_motor_{m}_velocity' in df.columns
            ]
            if other_vel_cols:
                df.loc[mask, f'data_motor_{mid}_other_vel_sum'] = (
                    df.loc[mask, other_vel_cols].fillna(0).sum(axis=1).values
                )

        # count of other motors that are 'moving'
        for mid in range(1, 7):
            other_mv_cols = [
                f'data_motor_{m}_is_moving' for m in range(1, 7)
                if m != mid and f'data_motor_{m}_is_moving' in df.columns
            ]
            if other_mv_cols:
                df.loc[mask, f'data_motor_{mid}_other_moving_count'] = (
                    df.loc[mask, other_mv_cols].fillna(0).sum(axis=1).values
                )

    df.fillna(0, inplace=True)
    return df


print('Feature engineering functions defined')

Feature engineering functions defined


## § 5 · Injection pool functions

### `make_injection_pool(train_df_raw, rng)`
For each motor:
- Find sequences that are entirely normal for that motor.
- Create `n` synthetic sequences by picking a random normal sequence,
  then injecting a rise-decay temperature pattern at a random location.
- Other motors' labels are set to 0 (synthetic fault is motor-specific).
- Sequences are given unique IDs like `syn_{motor_id}_{i}`.

Returns `dict {motor_id: raw_DataFrame}`.  Features are built *after* injection.

In [5]:
# Per-motor injection parameters
PER_MOTOR_INJECT = {
    1: dict(n=4,  peak_low=2, peak_high=10, dur_low=120, dur_high=400),
    2: dict(n=4,  peak_low=2, peak_high=10, dur_low=120, dur_high=400),
    3: dict(n=16, peak_low=1, peak_high=4,  dur_low=40,  dur_high=500),  # subtle
    4: dict(n=4,  peak_low=2, peak_high=10, dur_low=120, dur_high=400),
    5: dict(n=16, peak_low=1, peak_high=4,  dur_low=40,  dur_high=500),  # subtle
    6: dict(n=4,  peak_low=2, peak_high=10, dur_low=120, dur_high=400),
}


def _inject_rise_decay(temp_series: pd.Series, label_series: pd.Series,
                        start: int, duration: int, peak: float) -> pd.Series:
    """
    Inject a rise-decay temperature episode at position [start, start+duration).
    Returns modified temperature Series.
    """
    temp = temp_series.copy()
    n    = len(temp)
    end  = min(start + duration, n)
    dur  = end - start
    if dur <= 0:
        return temp

    n_rise = dur // 3
    n_fall = dur - n_rise
    t0     = float(temp.iloc[start])
    peak_t = t0 + peak

    rise = np.linspace(t0, peak_t, n_rise + 1)[:-1]
    fall = np.linspace(peak_t, t0,  n_fall)
    bump = np.concatenate([rise, fall])

    iloc_positions = list(range(start, end))
    temp.iloc[iloc_positions] = bump
    return temp


def make_injection_pool(train_df_raw: pd.DataFrame,
                        rng: np.random.Generator = None) -> dict:
    """
    Build per-motor injection pools from raw (no feature) training data.

    Returns
    -------
    dict {motor_id: DataFrame}  — each DataFrame is a stack of synthetic
    sequences, RAW (features not yet built).
    """
    if rng is None:
        rng = np.random.default_rng(42)

    pool = {}
    all_sequences = train_df_raw['test_condition'].unique()

    for mid in range(1, 7):
        cfg       = PER_MOTOR_INJECT[mid]
        n_syn     = cfg['n']
        label_col = f'data_motor_{mid}_label'
        temp_col  = f'data_motor_{mid}_temperature'

        if label_col not in train_df_raw.columns:
            continue

        # Find sequences that are entirely normal for this motor
        normal_seqs = []
        for seq in all_sequences:
            seq_df = train_df_raw[train_df_raw['test_condition'] == seq]
            if (seq_df[label_col] == 1).sum() == 0:
                normal_seqs.append(seq)

        if len(normal_seqs) == 0:
            print(f'  Motor {mid}: no normal sequences found, skipping injection.')
            continue

        syn_dfs = []
        for i in range(n_syn):
            # Pick a random normal sequence
            chosen_seq  = normal_seqs[int(rng.integers(0, len(normal_seqs)))]
            base_df     = train_df_raw[train_df_raw['test_condition'] == chosen_seq].copy()
            base_df     = base_df.reset_index(drop=True)
            seq_len     = len(base_df)

            # Random duration and start
            duration = int(rng.integers(cfg['dur_low'], cfg['dur_high'] + 1))
            peak     = float(rng.uniform(cfg['peak_low'], cfg['peak_high']))
            max_start = max(1, seq_len - duration)
            start    = int(rng.integers(0, max_start))

            # Inject temperature rise-decay
            if temp_col in base_df.columns:
                base_df[temp_col] = _inject_rise_decay(
                    base_df[temp_col], base_df[label_col], start, duration, peak
                )

            # Set labels for this motor in the injected window
            base_df[label_col] = 0
            end = min(start + duration, seq_len)
            base_df.loc[start:end - 1, label_col] = 1

            # Void other motors' labels (synthetic fault only for target motor)
            for other_mid in range(1, 7):
                if other_mid == mid:
                    continue
                other_lc = f'data_motor_{other_mid}_label'
                if other_lc in base_df.columns:
                    base_df[other_lc] = 0

            # Unique test_condition ID
            base_df['test_condition'] = f'syn_{mid}_{i}'
            syn_dfs.append(base_df)

        if syn_dfs:
            pool[mid] = pd.concat(syn_dfs, ignore_index=True)
            n1_total  = (pool[mid][label_col] == 1).sum()
            print(f'  Motor {mid}: {n_syn} synthetic sequences, {n1_total} fault rows')

    return pool


print('Injection pool functions defined')

Injection pool functions defined


## § 6 · Model config, rate functions, Viterbi, sample weights

### Model family
- Motors 1–4 → RandomForest (RF)
- Motors 5–6 → HistGradientBoosting (HGB)

### Sample weights with prevalence fade
`compute_weights` fades the class imbalance weight down as prevalence rises — avoids over-weighting when data is balanced.

### Rate-matched prediction
Instead of sweeping probability thresholds, find the **rate** (fraction of positive predictions)
that maximises F1 on OOF, then apply the same rate to test predictions.

### Viterbi episode decoding (motors 2 and 6)
2-state HMM in log-space.  `decode_to_rate` bisects on bias to hit a target positive rate.

In [6]:
# ── Algoritmos a comparar en cada motor ──────────────────────────────────────
ALGOS = ['LR', 'XGB', 'RF', 'HGB', 'IsoForest']


def _make_model(algo: str):
    """Instantiate a fresh estimator."""
    if algo == 'LR':
        return Pipeline([
            ('sc', StandardScaler()),
            ('lr', LogisticRegression(C=1.0, random_state=42, max_iter=1000)),
        ])
    elif algo == 'XGB':
        return xgb.XGBClassifier(
            n_estimators=150, learning_rate=0.05, max_depth=3,
            min_child_weight=10, eval_metric='logloss', verbosity=0,
            tree_method='hist', random_state=42,
        )
    elif algo == 'RF':
        return RandomForestClassifier(
            n_estimators=150, min_samples_leaf=10,
            class_weight='balanced_subsample', n_jobs=-1, random_state=42,
        )
    elif algo == 'HGB':
        return HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
            min_samples_leaf=100, l2_regularization=5.0, random_state=42,
        )
    elif algo == 'IsoForest':
        return IsolationForest(n_estimators=200, contamination='auto', random_state=42)
    else:
        raise ValueError(f'Unknown algo: {algo}')


def _fit(mdl, algo: str, X, y, sw):
    """Fit handling per-algorithm sample_weight signatures."""
    if algo == 'IsoForest':
        mdl.fit(X[y == 0])          # unsupervised: train on normal only
    elif algo == 'LR':
        mdl.fit(X, y, lr__sample_weight=sw)
    else:
        try:
            mdl.fit(X, y, sample_weight=sw)
        except TypeError:
            mdl.fit(X, y)
    return mdl


def _score(mdl, algo: str, X) -> 'np.ndarray':
    """Fault probability / anomaly score (higher = more likely fault)."""
    if algo == 'IsoForest':
        return (-mdl.decision_function(X)).astype(np.float32)
    return mdl.predict_proba(X)[:, 1].astype(np.float32)


# ── Sample weights (prevalence-aware sqrt-ratio, fades toward 1 at high prev) ─
WEIGHT_FADE_PREVALENCE = 0.15

def compute_weights(y: 'np.ndarray') -> 'np.ndarray':
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    if n0 == 0 or n1 == 0:
        return np.ones(len(y), dtype=np.float32)
    prev  = n1 / len(y)
    raw_w = np.sqrt(n0 / n1)
    fade  = min(1.0, prev / WEIGHT_FADE_PREVALENCE)
    eff_w = 1.0 + (raw_w - 1.0) * (1.0 - fade)
    return np.where(y == 1, eff_w, 1.0).astype(np.float32)


# ── Rate-matched prediction ───────────────────────────────────────────────────
def rate_matched_threshold(scores: 'np.ndarray', target_rate: float) -> float:
    s     = np.sort(scores)[::-1]
    n_pos = max(1, int(target_rate * len(s)))
    return float(s[n_pos]) if n_pos < len(s) else 0.0


def compute_additional_prevalence(df: 'pd.DataFrame', motor_id: int) -> float:
    col = f'data_motor_{motor_id}_label'
    if col not in df.columns or len(df) == 0:
        return 0.01
    return float((df[col] == 1).sum()) / len(df)


def find_oof_optimal_rate(oof_scores: 'np.ndarray', oof_labels: 'np.ndarray') -> float:
    best_f1, best_rate = -1.0, 0.02
    for rate in np.linspace(0.001, 0.30, 100):
        thr   = rate_matched_threshold(oof_scores, rate)
        preds = (oof_scores >= thr).astype(int)
        f1    = f1_score(oof_labels, preds, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_rate = f1, rate
    return best_rate


# ── Viterbi episode decoding (for long-fault motors) ─────────────────────────
EPISODE_MOTORS = {2, 6}


def mean_episode_length(labels: 'np.ndarray') -> float:
    lengths, cnt = [], 0
    for l in labels:
        if l == 1:
            cnt += 1
        elif cnt > 0:
            lengths.append(cnt); cnt = 0
    if cnt > 0:
        lengths.append(cnt)
    return float(np.mean(lengths)) if lengths else 50.0


def viterbi_path(prob: 'np.ndarray', a11: float = 0.99,
                 a00: float = 0.999, bias: float = 0.0) -> 'np.ndarray':
    n    = len(prob)
    p1   = np.clip(prob + bias, 1e-6, 1.0 - 1e-6)
    p0   = 1.0 - p1
    emit = np.stack([p0, p1], axis=1)
    logA = np.log(np.array([[a00, 1.0 - a00], [1.0 - a11, a11]]) + 1e-10)
    delta = np.full((n, 2), -np.inf)
    psi   = np.zeros((n, 2), dtype=int)
    delta[0] = np.log(emit[0] + 1e-10)
    for t in range(1, n):
        for s in range(2):
            scores     = delta[t - 1] + logA[:, s]
            psi[t, s]  = int(np.argmax(scores))
            delta[t, s] = float(np.max(scores)) + float(np.log(emit[t, s] + 1e-10))
    path = np.zeros(n, dtype=int)
    path[-1] = int(np.argmax(delta[-1]))
    for t in range(n - 2, -1, -1):
        path[t] = psi[t + 1, path[t + 1]]
    return path


def _decode_all(scores: 'np.ndarray', seq_ids: 'np.ndarray',
                a11: float, a00: float, bias: float = 0.0) -> 'np.ndarray':
    out = np.zeros(len(scores), dtype=int)
    for sid in np.unique(seq_ids):
        m = seq_ids == sid
        out[m] = viterbi_path(scores[m], a11=a11, a00=a00, bias=bias)
    return out


def decode_to_rate(scores: 'np.ndarray', seq_ids: 'np.ndarray',
                   target_rate: float, a11: float, a00: float) -> 'np.ndarray':
    lo, hi = -10.0, 10.0
    for _ in range(50):
        mid = (lo + hi) / 2.0
        if _decode_all(scores, seq_ids, a11, a00, bias=mid).mean() < target_rate:
            lo = mid
        else:
            hi = mid
    return _decode_all(scores, seq_ids, a11, a00, bias=(lo + hi) / 2.0)


# ── Protected rates and cap ───────────────────────────────────────────────────
PROTECTED_RATES    = {3: 0.005, 5: 0.004, 6: 0.008}
TEST_RATE_CAP_MULT = 2.0

print('Config OK')
print(f'Algoritmos por motor: {ALGOS}')


Config OK
Algoritmos por motor: ['LR', 'XGB', 'RF', 'HGB', 'IsoForest']


## § 7a · Load base training + test data (raw, minimal preprocessing)

Uses `pre_processing_minimal` — physical clip + ffill only, no centering, no feature derivation.
Features will be built later in § 7c / § 7d.

In [7]:
print('=== Loading base training data (minimal preprocessing) ===')
train_raw = read_all_test_data_from_path(TRAIN_PATH, pre_processing_minimal, is_plot=False)
print(f'  Shape: {train_raw.shape}')
print(f'  Sequences: {train_raw["test_condition"].nunique()}')

print()
print('=== Loading test data (minimal preprocessing) ===')
test_raw = read_all_test_data_from_path(TEST_PATH, pre_processing_minimal, is_plot=False)
print(f'  Shape: {test_raw.shape}')
print(f'  Sequences: {test_raw["test_condition"].nunique()}')

print()
print('=== Label distribution (base training only) ===')
print(f'{"Motor":<8} {"Normal":>8} {"Fault":>8} {"% fault":>9}')
print('-' * 38)
for mid in range(1, 7):
    col = f'data_motor_{mid}_label'
    if col in train_raw.columns:
        c0  = (train_raw[col] == 0).sum()
        c1  = (train_raw[col] == 1).sum()
        pct = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
        flag = '  <- imbalanced' if pct < 1.0 else ''
        print(f'{mid:<8} {c0:>8} {c1:>8} {pct:>8.1f}%{flag}')

=== Loading base training data (minimal preprocessing) ===
  Shape: (39309, 26)
  Sequences: 23

=== Loading test data (minimal preprocessing) ===
  Shape: (14157, 26)
  Sequences: 8

=== Label distribution (base training only) ===
Motor      Normal    Fault   % fault
--------------------------------------
1           37960     1349      3.4%
2           32577     6732     17.1%
3           39182      127      0.3%  <- imbalanced
4           32570     6739     17.1%
5           39125      184      0.5%  <- imbalanced
6           37377     1932      4.9%


## § 7b · Load + trust-filter additional data

Scans `additional_data/` for all groups (skipping `_tmp_` dirs).
Each group is loaded with `pre_processing_minimal`, then passed through `trust_filter_additional`
which voids mislabelled fault sequences where the temperature signature is not distinguishable
from the normal baseline.

In [8]:
additional_dfs = []   # list of trust-filtered additional DataFrames
all_audit_rows = []   # audit rows from trust_filter_additional

if not os.path.exists(ADDITIONAL_BASE):
    print(f'additional_data/ not found at {ADDITIONAL_BASE} — skipping.')
else:
    grupos = sorted([
        d for d in os.listdir(ADDITIONAL_BASE)
        if os.path.isdir(os.path.join(ADDITIONAL_BASE, d))
        and not d.startswith('_tmp')
    ])
    print(f'Found {len(grupos)} groups in additional_data/')

    for grupo in grupos:
        gpath = os.path.join(ADDITIONAL_BASE, grupo)
        print(f'\n  [{grupo}]')
        raw_g = _load_group_raw(gpath, grupo)
        if raw_g is None:
            print('    -> skipped (no xlsx or no valid sequences)')
            continue
        print(f'    Loaded: {raw_g.shape[0]} rows, {raw_g["test_condition"].nunique()} sequences')

        filtered_g, audit_g = trust_filter_additional(raw_g, temp_rise_margin=1.0)
        if len(audit_g) > 0:
            voided_blocks = (audit_g['voided'] > 0).sum()
            kept_blocks   = (audit_g['kept'] > 0).sum()
            print(f'    Trust-filter: {kept_blocks} blocks kept, {voided_blocks} blocks voided')
        all_audit_rows.append(audit_g)
        additional_dfs.append(filtered_g)

    if additional_dfs:
        additional_df = pd.concat(additional_dfs, ignore_index=True)
        print(f'\n  Combined additional_data: {additional_df.shape[0]} rows, '
              f'{additional_df["test_condition"].nunique()} sequences')

        print()
        print('  Label distribution in trust-filtered additional_data:')
        print(f'  {"Motor":<8} {"Normal":>8} {"Fault":>8} {"% fault":>9}')
        print('  ' + '-' * 38)
        for mid in range(1, 7):
            col = f'data_motor_{mid}_label'
            if col in additional_df.columns:
                c0  = (additional_df[col] == 0).sum()
                c1  = (additional_df[col] == 1).sum()
                pct = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
                print(f'  {mid:<8} {c0:>8} {c1:>8} {pct:>8.2f}%')
    else:
        additional_df = pd.DataFrame()
        print('\nNo additional data loaded.')

audit_df = pd.concat(all_audit_rows, ignore_index=True) if all_audit_rows else pd.DataFrame()
print('\nDone.')

Found 3 groups in additional_data/

  [additional_data_20240524_group_6]
    Loaded: 26086 rows, 13 sequences
    Trust-filter: 8 blocks kept, 0 blocks voided

  [additional_training_data_group_1]
    Loaded: 27592 rows, 13 sequences
    Trust-filter: 4 blocks kept, 0 blocks voided

  [additional_training_data_group_7]
    Loaded: 6420 rows, 10 sequences
    Trust-filter: 10 blocks kept, 0 blocks voided

  Combined additional_data: 60098 rows, 36 sequences

  Label distribution in trust-filtered additional_data:
  Motor      Normal    Fault   % fault
  --------------------------------------
  1           57632     2466     4.10%
  2           58734     1364     2.27%
  3           59391      707     1.18%
  4           58163     1935     3.22%
  5           58660     1438     2.39%
  6           59073     1025     1.71%

Done.


## § 7c · Build features on combined training data

Concatenates base training data + trust-filtered additional data,
then runs `build_features_for_df` to add all engineered features.
The result is stored in `train_feat_df`.

In [9]:
print('Combining base training + additional data ...')
if len(additional_df) > 0:
    combined_raw = pd.concat([train_raw, additional_df], ignore_index=True)
else:
    combined_raw = train_raw.copy()

print(f'Combined raw shape: {combined_raw.shape}  '
      f'({combined_raw["test_condition"].nunique()} sequences)')
print()
print('Building features for training data ...')
train_feat_df = build_features_for_df(combined_raw)
print(f'Train feature DataFrame shape: {train_feat_df.shape}')
print('Done.')

Combining base training + additional data ...
Combined raw shape: (99407, 26)  (59 sequences)

Building features for training data ...
  Building features for 59 sequences ...
  ... 10/59
  ... 20/59
  ... 30/59
  ... 40/59
  ... 50/59
  ... 59/59
Train feature DataFrame shape: (99407, 173)
Done.


## § 7d · Build features on test data

Applies the same feature engineering pipeline to the test set.
Stored in `test_feat_df`.

In [10]:
print('Building features for test data ...')
test_feat_df = build_features_for_df(test_raw)
print(f'Test feature DataFrame shape: {test_feat_df.shape}')
print(f'Test sequences: {test_feat_df["test_condition"].nunique()}')
print('Done.')

Building features for test data ...
  Building features for 8 sequences ...
  ... 8/8
Test feature DataFrame shape: (14157, 173)
Test sequences: 8
Done.


## § 7e · Build injection pool

1. `make_injection_pool` creates synthetic fault sequences from the **raw** training data.
2. `build_features_for_df` is applied to each motor's injection pool to get `injection_pool_feat`.

The injection pool is built from raw data (before feature engineering) and features are computed
afterwards to maintain consistency with the training pipeline.

In [11]:
print('Building raw injection pool from training data ...')
rng_global  = np.random.default_rng(42)
injection_pool_raw  = make_injection_pool(train_raw, rng=rng_global)

print()
print('Building features for each motor injection pool ...')
injection_pool_feat = {}
for mid, raw_pool in injection_pool_raw.items():
    print(f'  Motor {mid} ({len(raw_pool)} rows, {raw_pool["test_condition"].nunique()} sequences):')
    feat_pool = build_features_for_df(raw_pool)
    injection_pool_feat[mid] = feat_pool
    print(f'    -> features built, shape: {feat_pool.shape}')

print('\nInjection pool ready.')

Building raw injection pool from training data ...
  Motor 1: 4 synthetic sequences, 1043 fault rows
  Motor 2: 4 synthetic sequences, 1041 fault rows
  Motor 3: 16 synthetic sequences, 3704 fault rows
  Motor 4: 4 synthetic sequences, 991 fault rows
  Motor 5: 16 synthetic sequences, 2752 fault rows
  Motor 6: 4 synthetic sequences, 811 fault rows

Building features for each motor injection pool ...
  Motor 1 (6132 rows, 4 sequences):
  Building features for 4 sequences ...
  ... 4/4
    -> features built, shape: (6132, 173)
  Motor 2 (5533 rows, 4 sequences):
  Building features for 4 sequences ...
  ... 4/4
    -> features built, shape: (5533, 173)
  Motor 3 (12519 rows, 16 sequences):
  Building features for 16 sequences ...
  ... 10/16
  ... 16/16
    -> features built, shape: (12519, 173)
  Motor 4 (7637 rows, 4 sequences):
  Building features for 4 sequences ...
  ... 4/4
    -> features built, shape: (7637, 173)
  Motor 5 (17140 rows, 16 sequences):
  Building features for 16 s

## § 8 · Funcion `train_motor()` — compara 5 algoritmos por motor

Para cada motor se evaluan con **GroupKFold (5 pliegues)** los algoritmos:

| Algo | Tipo | Caracteristica |
|---|---|---|
| **LR** | Supervisado | Lineal, robusto con pocos datos |
| **XGBoost** | Supervisado | Boosting, bueno con features tabulares |
| **RF** | Supervisado | Ensemble de arboles, estable |
| **HGB** | Supervisado | Gradient boosting con histogramas, rapido |
| **IsoForest** | No supervisado | Entrena solo sobre datos normales |

**Flujo por motor:**
1. Pool de entrenamiento: datos base + additional (trust-filtered) + inyeccion calibrada.
2. GroupKFold OOF para cada algoritmo → scores de anomalia/probabilidad.
3. Para cada algoritmo: encontrar rate optimo que maximiza F1 OOF, luego aplicar cap de prevalencia.
4. Ganador = mayor F1 OOF rate-matched.
5. Refit del ganador sobre pool completo.
6. Prediccion en test: rate matching; motores 2 y 6 con Viterbi HMM.

In [12]:
def _get_feat_cols(df: 'pd.DataFrame', motor_id: int) -> list:
    prefix = f'data_motor_{motor_id}_'
    cols   = [c for c in df.columns
              if c.startswith(prefix) and not c.endswith('_label')]
    if 'mean_voltage' in df.columns:
        cols.append('mean_voltage')
    return sorted(set(cols))


def _oof_scores_for(algo: str, X, y, g, sw, n_splits: int = 5) -> 'np.ndarray':
    """GroupKFold OOF scores for one algorithm."""
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(y), dtype=np.float32)
    for tr_idx, va_idx in gkf.split(X, y, groups=g):
        mdl = _make_model(algo)
        _fit(mdl, algo, X[tr_idx], y[tr_idx], sw[tr_idx])
        oof[va_idx] = _score(mdl, algo, X[va_idx])
    return oof


def train_motor(motor_id: int):
    SEP  = '=' * 70
    LINE = '-' * 70
    print(f'\n{SEP}')
    print(f'  MOTOR {motor_id}')
    print(f'{SEP}')

    label_col = f'data_motor_{motor_id}_label'
    feat_cols = _get_feat_cols(train_feat_df, motor_id)
    if not feat_cols or label_col not in train_feat_df.columns:
        print('  SKIP'); return
    print(f'  Features: {len(feat_cols)} columnas')

    # ── Training pool: original + filtered additional + injection ─────────────
    base  = train_feat_df.dropna(subset=[label_col]).copy()
    parts = [base]
    if motor_id in injection_pool_feat and len(injection_pool_feat[motor_id]) > 0:
        inj = injection_pool_feat[motor_id].copy()
        for c in feat_cols:
            if c not in inj.columns:
                inj[c] = 0.0
        parts.append(inj)
        print(f'  Injection pool: {len(inj)} rows  '
              f'({inj["test_condition"].nunique()} syn seqs)')

    pool = pd.concat(parts, ignore_index=True)
    pool[feat_cols] = pool[feat_cols].fillna(0)

    X  = pool[feat_cols].values.astype(np.float32)
    y  = pool[label_col].values.astype(int)
    g  = pool['test_condition'].values
    sw = compute_weights(y)

    n0, n1 = (y == 0).sum(), (y == 1).sum()
    print(f'  Pool: {len(y)} filas  (normal={n0}, fallo={n1}, {100*n1/len(y):.2f}%)')

    # Prevalencia en additional_df (trust-filtered) para el rate cap
    _add_df  = additional_df if len(additional_df) > 0 else pd.DataFrame()
    add_prev = compute_additional_prevalence(_add_df, motor_id)
    print(f'  Prevalencia en additional (trust-filtered): {add_prev:.4f}')
    if motor_id in PROTECTED_RATES:
        print(f'  Rate protegida: {PROTECTED_RATES[motor_id]}')

    # ── GroupKFold OOF para cada algoritmo ────────────────────────────────────
    print(f'\n  {"Algo":<12}  {"OOF F1":>9}  {"OOF rate":>10}  {"Eff rate":>10}')
    print(f'  {"-"*48}')

    results = {}
    for algo in ALGOS:
        oof_sc   = _oof_scores_for(algo, X, y, g, sw)
        oof_rate = find_oof_optimal_rate(oof_sc, y)

        if motor_id in PROTECTED_RATES:
            eff_rate = PROTECTED_RATES[motor_id]
        else:
            cap      = TEST_RATE_CAP_MULT * add_prev if add_prev > 0 else oof_rate
            eff_rate = min(oof_rate, cap)

        thr    = rate_matched_threshold(oof_sc, eff_rate)
        oof_f1 = f1_score(y, (oof_sc >= thr).astype(int), pos_label=1, zero_division=0)

        results[algo] = dict(oof_f1=oof_f1, oof_rate=oof_rate,
                             eff_rate=eff_rate, oof_sc=oof_sc)

        cur_best = max(r['oof_f1'] for r in results.values())
        mark = '  <- best' if oof_f1 == cur_best else ''
        print(f'  {algo:<12}  {oof_f1:>9.4f}  {oof_rate:>10.4f}  {eff_rate:>10.4f}{mark}')

    # ── Ganador ───────────────────────────────────────────────────────────────
    best_algo = max(results, key=lambda a: results[a]['oof_f1'])
    best      = results[best_algo]
    eff_rate  = best['eff_rate']
    oof_f1    = best['oof_f1']

    rank_str = '  >  '.join(
        f'{a}: {results[a]["oof_f1"]:.4f}'
        for a in sorted(results, key=lambda x: results[x]['oof_f1'], reverse=True)
    )
    print(f'\n  GANADOR Motor {motor_id}: {best_algo}  '
          f'OOF F1={oof_f1:.4f}  rate={eff_rate:.4f}')
    print(f'  Ranking: {rank_str}')

    # ── Refit ganador sobre el pool completo ──────────────────────────────────
    print(f'  Refitting {best_algo} sobre pool completo...')
    final_mdl = _make_model(best_algo)
    _fit(final_mdl, best_algo, X, y, sw)

    # ── Prediccion sobre test set ─────────────────────────────────────────────
    test_cols = [c for c in feat_cols if c in test_feat_df.columns]
    X_test    = test_feat_df[test_cols].fillna(0).values.astype(np.float32)
    test_sc   = _score(final_mdl, best_algo, X_test)

    if motor_id in EPISODE_MOTORS:
        mean_ep    = mean_episode_length(y)
        a11        = max(0.90, 1.0 - 1.0 / max(mean_ep, 5.0))
        a00        = min(0.9999, max(0.990, 1.0 - add_prev * 2))
        seq_test   = test_feat_df['test_condition'].values
        test_preds = decode_to_rate(test_sc, seq_test, eff_rate, a11, a00)
        print(f'  Viterbi: a11={a11:.4f}  a00={a00:.4f}  mean_ep={mean_ep:.1f}')
    else:
        thr_test   = rate_matched_threshold(test_sc, eff_rate)
        test_preds = (test_sc >= thr_test).astype(int)

    pos = int(test_preds.sum())
    print(f'  Test positivos: {pos}/{len(test_preds)} ({100*pos/len(test_preds):.2f}%)')

    models[motor_id] = dict(
        algo       = best_algo,
        model      = final_mdl,
        eff_rate   = eff_rate,
        oof_f1     = oof_f1,
        test_preds = test_preds,
        ranking    = {a: results[a]['oof_f1'] for a in results},
    )
    val_f1_per_motor[motor_id] = oof_f1
    print(f'  Motor {motor_id} listo.')


models           = {}
val_f1_per_motor = {}
print('train_motor() v4 definida — compara', ALGOS, 'en cada motor')

train_motor() v4 definida — compara ['LR', 'XGB', 'RF', 'HGB', 'IsoForest'] en cada motor


## § 9 · Motor 1

Algorithm: RandomForest.  Shape features included.

In [13]:
train_motor(1)


  MOTOR 1
  Features: 29 columnas
  Injection pool: 6132 rows  (4 syn seqs)
  Pool: 105539 filas  (normal=100681, fallo=4858, 4.60%)
  Prevalencia en additional (trust-filtered): 0.0410

  Algo             OOF F1    OOF rate    Eff rate
  ------------------------------------------------
  LR               0.2043      0.0825      0.0821  <- best
  XGB              0.3141      0.0705      0.0705  <- best
  RF               0.2794      0.1550      0.0821
  HGB              0.3427      0.0614      0.0614  <- best
  IsoForest        0.0947      0.2819      0.0821

  GANADOR Motor 1: HGB  OOF F1=0.3427  rate=0.0614
  Ranking: HGB: 0.3427  >  XGB: 0.3141  >  RF: 0.2794  >  LR: 0.2043  >  IsoForest: 0.0947
  Refitting HGB sobre pool completo...
  Test positivos: 870/14157 (6.15%)
  Motor 1 listo.


## § 10 · Motor 2

Algorithm: RandomForest.  Shape features included.  Viterbi episode decoding applied.

In [14]:
train_motor(2)


  MOTOR 2
  Features: 29 columnas
  Injection pool: 5533 rows  (4 syn seqs)
  Pool: 104940 filas  (normal=95803, fallo=9137, 8.71%)
  Prevalencia en additional (trust-filtered): 0.0227

  Algo             OOF F1    OOF rate    Eff rate
  ------------------------------------------------
  LR               0.1312      0.0916      0.0454  <- best
  XGB              0.2690      0.0403      0.0403  <- best
  RF               0.2517      0.0312      0.0312
  HGB              0.2522      0.0584      0.0454
  IsoForest        0.1903      0.2003      0.0454

  GANADOR Motor 2: XGB  OOF F1=0.2690  rate=0.0403
  Ranking: XGB: 0.2690  >  HGB: 0.2522  >  RF: 0.2517  >  IsoForest: 0.1903  >  LR: 0.1312
  Refitting XGB sobre pool completo...
  Viterbi: a11=0.9989  a00=0.9900  mean_ep=913.7
  Test positivos: 622/14157 (4.39%)
  Motor 2 listo.


## § 11 · Motor 3

Algorithm: RandomForest.  Shape features included.  Protected rate = 0.005.

In [15]:
train_motor(3)


  MOTOR 3
  Features: 29 columnas
  Injection pool: 12519 rows  (16 syn seqs)
  Pool: 111926 filas  (normal=107388, fallo=4538, 4.05%)
  Prevalencia en additional (trust-filtered): 0.0118
  Rate protegida: 0.005

  Algo             OOF F1    OOF rate    Eff rate
  ------------------------------------------------
  LR               0.0165      0.0765      0.0050  <- best
  XGB              0.2239      0.0312      0.0050  <- best
  RF               0.2197      0.0372      0.0050
  HGB              0.2197      0.0342      0.0050
  IsoForest        0.0098      0.3000      0.0050

  GANADOR Motor 3: XGB  OOF F1=0.2239  rate=0.0050
  Ranking: XGB: 0.2239  >  RF: 0.2197  >  HGB: 0.2197  >  LR: 0.0165  >  IsoForest: 0.0098
  Refitting XGB sobre pool completo...
  Test positivos: 71/14157 (0.50%)
  Motor 3 listo.


## § 12 · Motor 4

Algorithm: RandomForest.  No shape features.

In [16]:
train_motor(4)


  MOTOR 4
  Features: 27 columnas
  Injection pool: 7637 rows  (4 syn seqs)
  Pool: 107044 filas  (normal=97379, fallo=9665, 9.03%)
  Prevalencia en additional (trust-filtered): 0.0322

  Algo             OOF F1    OOF rate    Eff rate
  ------------------------------------------------
  LR               0.1698      0.1127      0.0644  <- best
  XGB              0.1618      0.0976      0.0644
  RF               0.1912      0.2154      0.0644  <- best
  HGB              0.1920      0.0554      0.0554  <- best
  IsoForest        0.1681      0.2185      0.0644

  GANADOR Motor 4: HGB  OOF F1=0.1920  rate=0.0554
  Ranking: HGB: 0.1920  >  RF: 0.1912  >  LR: 0.1698  >  IsoForest: 0.1681  >  XGB: 0.1618
  Refitting HGB sobre pool completo...
  Test positivos: 784/14157 (5.54%)
  Motor 4 listo.


## § 13 · Motor 5

Algorithm: HistGradientBoosting.  Shape features included.  Protected rate = 0.004.

In [17]:
train_motor(5)


  MOTOR 5
  Features: 29 columnas
  Injection pool: 17140 rows  (16 syn seqs)
  Pool: 116547 filas  (normal=112173, fallo=4374, 3.75%)
  Prevalencia en additional (trust-filtered): 0.0239
  Rate protegida: 0.004

  Algo             OOF F1    OOF rate    Eff rate
  ------------------------------------------------
  LR               0.1562      0.0342      0.0040  <- best
  XGB              0.2066      0.0252      0.0040  <- best
  RF               0.1929      0.0252      0.0040
  HGB              0.1929      0.0252      0.0040
  IsoForest        0.0235      0.2758      0.0040

  GANADOR Motor 5: XGB  OOF F1=0.2066  rate=0.0040
  Ranking: XGB: 0.2066  >  RF: 0.1929  >  HGB: 0.1929  >  LR: 0.1562  >  IsoForest: 0.0235
  Refitting XGB sobre pool completo...
  Test positivos: 57/14157 (0.40%)
  Motor 5 listo.


## § 14 · Motor 6

Algorithm: HistGradientBoosting.  No shape features.  Viterbi episode decoding + protected rate = 0.008.

In [18]:
train_motor(6)


  MOTOR 6
  Features: 27 columnas
  Injection pool: 10860 rows  (4 syn seqs)
  Pool: 110267 filas  (normal=106499, fallo=3768, 3.42%)
  Prevalencia en additional (trust-filtered): 0.0171
  Rate protegida: 0.008

  Algo             OOF F1    OOF rate    Eff rate
  ------------------------------------------------
  LR               0.2713      0.0191      0.0080  <- best
  XGB              0.3423      0.0312      0.0080  <- best
  RF               0.2421      0.0433      0.0080
  HGB              0.3298      0.0282      0.0080
  IsoForest        0.2232      0.0312      0.0080

  GANADOR Motor 6: XGB  OOF F1=0.3423  rate=0.0080
  Ranking: XGB: 0.3423  >  HGB: 0.3298  >  LR: 0.2713  >  RF: 0.2421  >  IsoForest: 0.2232
  Refitting XGB sobre pool completo...
  Viterbi: a11=0.9960  a00=0.9900  mean_ep=251.2
  Test positivos: 115/14157 (0.81%)
  Motor 6 listo.


## § 15 · Summary

Shows model type, target rate, OOF F1 per motor, and a text bar chart.

In [19]:
SEP  = '=' * 70
LINE = '-' * 70

print(SEP)
print('  RESUMEN FINAL v4 — MEJOR ALGORITMO POR MOTOR')
print(SEP)
print(f'  {"Motor":<8} {"Algoritmo":<12} {"OOF F1":>9}  Barra')
print('  ' + LINE)

scored = {}
for mid in range(1, 7):
    info = models.get(mid)
    if info is None:
        print(f'  Motor {mid}   -- no entrenado'); continue
    algo = info['algo']
    f1   = info['oof_f1']
    rate = info['eff_rate']
    scored[mid] = f1
    bar  = '#' * int(f1 * 30)
    rank = info.get('ranking', {})
    rank_str = '  >  '.join(
        f'{a}: {rank[a]:.3f}'
        for a in sorted(rank, key=rank.get, reverse=True)
    )
    print(f'  Motor {mid}   {algo:<12} {f1:>9.4f}  {bar}')
    print(f'           rate={rate:.4f}  |  {rank_str}')
    print()

if scored:
    gf1 = sum(scored.values()) / len(scored)
    print('  ' + LINE)
    print(f'  Global mean OOF F1 ({len(scored)} motores): {gf1:.4f}')
print(SEP)


  RESUMEN FINAL v4 — MEJOR ALGORITMO POR MOTOR
  Motor    Algoritmo       OOF F1  Barra
  ----------------------------------------------------------------------
  Motor 1   HGB             0.3427  ##########
           rate=0.0614  |  HGB: 0.343  >  XGB: 0.314  >  RF: 0.279  >  LR: 0.204  >  IsoForest: 0.095

  Motor 2   XGB             0.2690  ########
           rate=0.0403  |  XGB: 0.269  >  HGB: 0.252  >  RF: 0.252  >  IsoForest: 0.190  >  LR: 0.131

  Motor 3   XGB             0.2239  ######
           rate=0.0050  |  XGB: 0.224  >  RF: 0.220  >  HGB: 0.220  >  LR: 0.016  >  IsoForest: 0.010

  Motor 4   HGB             0.1920  #####
           rate=0.0554  |  HGB: 0.192  >  RF: 0.191  >  LR: 0.170  >  IsoForest: 0.168  >  XGB: 0.162

  Motor 5   XGB             0.2066  ######
           rate=0.0040  |  XGB: 0.207  >  RF: 0.193  >  HGB: 0.193  >  LR: 0.156  >  IsoForest: 0.024

  Motor 6   XGB             0.3423  ##########
           rate=0.0080  |  XGB: 0.342  >  HGB: 0.330  >  

## § 16 · Submission

Assembles the final submission CSV from per-motor test predictions.

`EXCLUDE_MOTOR = N` → forces that motor's predictions to `-1` (Kaggle trick to avoid spending a real submission).
`EXCLUDE_MOTOR = -1` → full submission (all motors).

In [20]:
EXCLUDE_MOTOR = None   # None = submission final  |  1-6 = truco Kaggle

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
final_sub  = sample_sub.copy()
for mid in range(1, 7):
    final_sub[f'data_motor_{mid}_label'] = 0

# Alinear test_preds con sample_sub usando test_condition + posicion dentro de la secuencia
test_seq_col = test_feat_df['test_condition'].values

for mid in range(1, 7):
    info = models.get(mid)
    if info is None:
        continue
    test_preds = info['test_preds']
    seq_pos    = {}   # {seq_id: contador de fila dentro de esa secuencia}
    for i, (seq_id, pred) in enumerate(zip(test_seq_col, test_preds)):
        pos_within = seq_pos.get(seq_id, 0)
        seq_mask   = sample_sub['test_condition'] == seq_id
        idxs       = sample_sub.index[seq_mask].tolist()
        if pos_within < len(idxs):
            final_sub.loc[idxs[pos_within], f'data_motor_{mid}_label'] = int(pred)
        seq_pos[seq_id] = pos_within + 1

for mid in range(1, 7):
    final_sub[f'data_motor_{mid}_label'] = final_sub[f'data_motor_{mid}_label'].astype(int)

if EXCLUDE_MOTOR is not None:
    print(f'Truco Kaggle: motor {EXCLUDE_MOTOR} -> -1')
    final_sub[f'data_motor_{EXCLUDE_MOTOR}_label'] = -1
    out = os.path.join(NOTEBOOK_ROOT, f'motor_excluded_{EXCLUDE_MOTOR}_v4.csv')
else:
    out = os.path.join(NOTEBOOK_ROOT, 'motor_submission_v4.csv')

final_sub.to_csv(out, index=False)
print(f'Guardado: {out}  shape={final_sub.shape}')
print()
print(f'{"Motor":<8} {"Algo":<12} {"0 (normal)":>12} {"1 (fallo)":>10} {"% fallo":>9}')
print('-' * 56)
for mid in range(1, 7):
    col  = f'data_motor_{mid}_label'
    algo = models.get(mid, {}).get('algo', '?')
    c0   = (final_sub[col] == 0).sum()
    c1   = (final_sub[col] == 1).sum()
    pct  = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
    print(f'{mid:<8} {algo:<12} {c0:>12} {c1:>10} {pct:>8.2f}%')
final_sub.head()


Guardado: c:\Users\oscar\Desktop\TDS INDUSTRY\motor_submission_v4.csv  shape=(14157, 8)

Motor    Algo           0 (normal)  1 (fallo)   % fallo
--------------------------------------------------------
1        HGB                 13287        870     6.15%
2        XGB                 13535        622     4.39%
3        XGB                 14086         71     0.50%
4        HGB                 13373        784     5.54%
5        XGB                 14100         57     0.40%
6        XGB                 14042        115     0.81%


,idx,data_motor_1_label,data_motor_2_label,data_motor_3_label,data_motor_4_label,data_motor_5_label,data_motor_6_label,test_condition
0,0,0,0,0,0,0,0,20240527_094865
1,1,0,0,0,0,0,0,20240527_094865
2,2,0,0,0,0,0,0,20240527_094865
3,3,0,0,0,0,0,0,20240527_094865
4,4,0,0,0,0,0,0,20240527_094865
